# 0. Data Collection

This notebook fetches all raw datasets required for the HDB Resale Price Prediction project.

**Data Sources:**
1. HDB Resale Transactions (4 datasets: 2000-2012, 2012-2014, 2015-2016, 2017 onwards) — data.gov.sg
2. MRT/LRT Stations — coordinates from LTA DataMall shapefile (official), opening dates from public records
3. Schools — data.gov.sg
4. HDB Property Information — data.gov.sg
5. Hawker Centres — data.gov.sg
6. HDB Car Park Information — data.gov.sg
7. Shopping Malls — curated list (86 malls)

All outputs are saved to `../data/raw/`.

In [14]:
%pip install geopandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [15]:
import sys
sys.path.append('..')

import pandas as pd
from src.data_collection import (
    fetch_datagov_csv, fetch_all_resale_transactions,
    get_mrt_stations, get_shopping_malls
)

## 1. HDB Resale Transactions

Four datasets spanning 2000 to present, fetched from data.gov.sg and concatenated:

| Period | Dataset ID |
|--------|-----------|
| 2000 – Feb 2012 | `d_43f493c6c50d54243cc1eab0df142d6a` |
| Mar 2012 – Dec 2014 | `d_2d5ff9ea31397b66239f245f57751537` |
| Jan 2015 – Dec 2016 | `d_ea9ed51da2787afaf8e51f827c304208` |
| Jan 2017 onwards | `d_8b84c4ee58e3cfc0ece0d773c8ca6abc` |

If the API fails, download CSVs manually from https://data.gov.sg and place them in `../data/raw/`.

In [16]:
df_resale = fetch_all_resale_transactions("../data/raw")

print(f"\nCombined shape: {df_resale.shape}")
print(f"Date range: {df_resale['month'].min()} to {df_resale['month'].max()}")
print(f"Columns: {list(df_resale.columns)}")
df_resale.head()


--- 2000_2012: already exists, loading from disk ---
Loaded 369651 rows from ../data/raw/resale_transactions_2000_2012.csv

--- 2012_2014: already exists, loading from disk ---
Loaded 52203 rows from ../data/raw/resale_transactions_2012_2014.csv

--- 2015_2016: already exists, loading from disk ---
Loaded 37153 rows from ../data/raw/resale_transactions_2015_2016.csv

--- 2017_onwards: already exists, loading from disk ---
Loaded 233712 rows from ../data/raw/resale_transactions_2017_onwards.csv

Combined: 692719 total rows saved to ../data/raw/resale_transactions.csv

Combined shape: (692719, 11)
Date range: 2000-01 to 2026-06
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease
0,2000-01,ANG MO KIO,3 ROOM,170,ANG MO KIO AVE 4,07 TO 09,69.0,Improved,1986,147000.0,NaN
1,2000-01,ANG MO KIO,3 ROOM,174,ANG MO KIO AVE 4,04 TO 06,61.0,Improved,1986,144000.0,NaN
2,2000-01,ANG MO KIO,3 ROOM,216,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,159000.0,NaN
3,2000-01,ANG MO KIO,3 ROOM,215,ANG MO KIO AVE 1,07 TO 09,73.0,New Generation,1976,167000.0,NaN
4,2000-01,ANG MO KIO,3 ROOM,218,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1976,163000.0,NaN


In [17]:
# Check for column differences across periods
# The 2000-2012 dataset may lack 'remaining_lease' column
print("Null counts per column:")
print(df_resale.isnull().sum())
print(f"\nTransactions per year:")
df_resale['month'] = df_resale['month'].astype(str)
df_resale['year'] = df_resale['month'].str[:4]
print(df_resale['year'].value_counts().sort_index())

Null counts per column:
month                       0
town                        0
flat_type                   0
block                       0
street_name                 0
storey_range                0
floor_area_sqm              0
flat_model                  0
lease_commence_date         0
resale_price                0
remaining_lease        421854
dtype: int64

Transactions per year:
year
2000    34862
2001    38055
2002    36098
2003    29003
2004    29112
2005    30045
2006    27427
2007    26982
2008    27262
2009    30482
2010    34854
2011    22281
2012    23198
2013    16097
2014    16096
2015    17780
2016    19373
2017    20509
2018    21561
2019    22186
2020    23333
2021    29087
2022    26720
2023    25754
2024    27832
2025    25085
2026    11645
Name: count, dtype: int64


## 2. MRT/LRT Stations

Coordinates sourced from the **LTA DataMall shapefile** (official, March 2026 release). Opening dates from LTA press releases and Wikipedia. The shapefile uses SVY21 projection — we convert polygon centroids to WGS84 lat/lng.

Unbuilt stations (Bukit Brown, Founders' Memorial) are assigned `2099-12-31` so they are never included in historical distance calculations.

In [18]:
SHAPEFILE_PATH = "../TrainStation_Mar2026/RapidTransitSystemStation.shp"

df_mrt = get_mrt_stations(SHAPEFILE_PATH)
df_mrt.to_csv("../data/raw/mrt_lrt_stations.csv", index=False)

print(f"Total stations: {len(df_mrt)}")
print(f"Unique stations: {df_mrt['station_name'].nunique()}")
print(f"MRT: {len(df_mrt[df_mrt['type'] == 'MRT'])}, LRT: {len(df_mrt[df_mrt['type'] == 'LRT'])}")
print(f"\nOpening date range: {df_mrt['opening_date'].min()} to {df_mrt['opening_date'].max()}")
df_mrt.head(10)

Total stations: 190
Unique stations: 186
MRT: 148, LRT: 42

Opening date range: 1987-11-07 00:00:00 to 2099-12-31 00:00:00


,station_name,type,lat,lng,opening_date
0,Coral Edge,LRT,1.393926,103.912564,2005-01-29
1,Hougang,MRT,1.371281,103.892348,2003-06-20
2,Bedok,MRT,1.324011,103.930173,1989-12-16
3,Punggol,LRT,1.405167,103.902393,2003-06-20
4,Sam Kee,LRT,1.409614,103.904831,2016-12-29
5,Jurong East,MRT,1.333209,103.742263,1990-03-10
6,Bukit Brown,MRT,1.333732,103.830712,2099-12-31
7,Tampines East,MRT,1.356213,103.954805,2017-10-21
8,Soo Teck,LRT,1.405210,103.897234,2005-01-29
9,Fajar,LRT,1.384533,103.770838,1999-11-06


In [19]:
# Stations opened during our data window (2017+) — these are the ones
# that would cause data leakage if used statically
recent_stations = df_mrt[df_mrt["opening_date"] >= "2017-01-01"].sort_values("opening_date")
print(f"Stations opened 2017 onwards: {len(recent_stations)}")
recent_stations[["station_name", "type", "opening_date"]]

Stations opened 2017 onwards: 49


,station_name,type,opening_date
127,Tuas Link,MRT,2017-06-18
156,Gul Circle,MRT,2017-06-18
133,Tuas Crescent,MRT,2017-06-18
134,Tuas West Road,MRT,2017-06-18
120,Geylang Bahru,MRT,2017-10-21
183,Rochor,MRT,2017-10-21
122,Bedok Reservoir,MRT,2017-10-21
77,Upper Changi,MRT,2017-10-21
71,Kaki Bukit,MRT,2017-10-21
128,Stevens,MRT,2017-10-21


## 3. Schools

Source: https://beta.data.gov.sg/collections/457/datasets/d_688b934f82c1059ed0a6993d2a829089/view

346 schools with SAP, autonomous, gifted, and IP indicators.

In [20]:
import os
if os.path.exists("../data/raw/schools.csv"):
    df_schools = pd.read_csv("../data/raw/schools.csv")
    print(f"Already exists, loaded {len(df_schools)} rows from disk")
else:
    df_schools = fetch_datagov_csv(
        dataset_id="d_688b934f82c1059ed0a6993d2a829089",
        save_path="../data/raw/schools.csv"
    )
print(f"Shape: {df_schools.shape}")
print(f"\nSchool levels: {df_schools['mainlevel_code'].value_counts().to_dict()}")
print(f"SAP schools: {(df_schools['sap_ind'] == 'Yes').sum()}")
print(f"Autonomous schools: {(df_schools['autonomous_ind'] == 'Yes').sum()}")
print(f"Gifted schools: {(df_schools['gifted_ind'] == 'Yes').sum()}")
print(f"IP schools: {(df_schools['ip_ind'] == 'Yes').sum()}")
df_schools.head()

Already exists, loaded 337 rows from disk
Shape: (337, 31)

School levels: {'PRIMARY': 179, 'SECONDARY (S1-S5)': 117, 'SECONDARY (S1-S4)': 16, 'JUNIOR COLLEGE': 10, 'MIXED LEVEL (S1-JC2)': 10, 'MIXED LEVEL (P1-S4)': 3, 'CENTRALISED INSTITUTE': 1, 'MIXED LEVEL (S1-S5, JC1-JC2)': 1}
SAP schools: 23
Autonomous schools: 28
Gifted schools: 16
IP schools: 19


,school_name,url_address,address,postal_code,telephone_no,telephone_no_2,fax_no,fax_no_2,email_address,mrt_desc,...,nature_code,session_code,mainlevel_code,sap_ind,autonomous_ind,gifted_ind,ip_ind,mothertongue1_code,mothertongue2_code,mothertongue3_code
0,ADMIRALTY PRIMARY SCHOOL,https://admiraltypri.moe.edu.sg/,11 WOODLANDS CIRCLE,738907,63620598,na,63627512,na,admiralty_ps@moe.edu.sg,Admiralty Station,...,CO-ED SCHOOL,FULL DAY,PRIMARY,No,No,No,No,CHINESE,MALAY,TAMIL
1,ADMIRALTY SECONDARY SCHOOL,http://www.admiraltysec.moe.edu.sg,31 WOODLANDS CRESCENT,737916,63651733,63654596,63652774,na,admiralty_ss@moe.edu.sg,ADMIRALTY MRT,...,CO-ED SCHOOL,SINGLE SESSION,SECONDARY (S1-S5),No,No,No,No,CHINESE,MALAY,TAMIL
2,AHMAD IBRAHIM PRIMARY SCHOOL,http://www.ahmadibrahimpri.moe.edu.sg,10 YISHUN STREET 11,768643,67592906,na,67592927,na,aips@moe.edu.sg,Yishun,...,CO-ED SCHOOL,SINGLE SESSION,PRIMARY,No,No,No,No,CHINESE,MALAY,TAMIL
3,AHMAD IBRAHIM SECONDARY SCHOOL,http://www.ahmadibrahimsec.moe.edu.sg,751 YISHUN AVENUE 7,768928,67585384,na,67557778,na,aiss@moe.edu.sg,"CANBERRA MRT, YISHUN MRT",...,CO-ED SCHOOL,SINGLE SESSION,SECONDARY (S1-S5),No,No,No,No,CHINESE,MALAY,TAMIL
4,AI TONG SCHOOL,http://www.aitong.moe.edu.sg,100 Bright Hill Drive,579646,64547672,na,64532726,na,aitong_sch@moe.edu.sg,Bishan MRT,...,CO-ED SCHOOL,SINGLE SESSION,PRIMARY,Yes,No,No,No,CHINESE,na,na


## 4. HDB Property Information

Source: https://beta.data.gov.sg/datasets/d_17f5382f26140b1fdae0ba2ef6239d2f/view

Building metadata: max floors, year completed, facility flags (residential, commercial, market/hawker, etc.).

In [21]:
if os.path.exists("../data/raw/hdb_property_info.csv"):
    df_property = pd.read_csv("../data/raw/hdb_property_info.csv")
    print(f"Already exists, loaded {len(df_property)} rows from disk")
else:
    df_property = fetch_datagov_csv(
        dataset_id="d_17f5382f26140b1fdae0ba2ef6239d2f",
        save_path="../data/raw/hdb_property_info.csv"
    )
print(f"Shape: {df_property.shape}")
print(f"Columns: {list(df_property.columns)}")
df_property.head()

Already exists, loaded 13289 rows from disk
Shape: (13289, 24)
Columns: ['blk_no', 'street', 'max_floor_lvl', 'year_completed', 'residential', 'commercial', 'market_hawker', 'miscellaneous', 'multistorey_carpark', 'precinct_pavilion', 'bldg_contract_town', 'total_dwelling_units', '1room_sold', '2room_sold', '3room_sold', '4room_sold', '5room_sold', 'exec_sold', 'multigen_sold', 'studio_apartment_sold', '1room_rental', '2room_rental', '3room_rental', 'other_room_rental']


,blk_no,street,max_floor_lvl,year_completed,residential,commercial,market_hawker,miscellaneous,multistorey_carpark,precinct_pavilion,...,3room_sold,4room_sold,5room_sold,exec_sold,multigen_sold,studio_apartment_sold,1room_rental,2room_rental,3room_rental,other_room_rental
0,1,BEACH RD,16,1970,Y,Y,N,N,N,N,...,138,1,2,0,0,0,0,0,0,0
1,1,BEDOK STH AVE 1,14,1975,Y,N,N,Y,N,N,...,204,0,2,0,0,0,0,0,0,0
2,1,CANTONMENT RD,2,2010,N,Y,N,N,N,N,...,0,0,0,0,0,0,0,0,0,0
3,1,CHAI CHEE RD,15,1982,Y,N,N,N,N,N,...,0,10,92,0,0,0,0,0,0,0
4,1,CHANGI VILLAGE RD,4,1975,Y,Y,N,N,N,N,...,54,0,1,0,0,0,0,0,0,0


## 5. Hawker Centres

Source: https://beta.data.gov.sg/collections/1445/datasets/d_4a086da0a5553be1d89383cd90d07ecd/view

NEA dataset of hawker centre locations with coordinates.

In [22]:
import requests
import time
import os

# Skip if already downloaded
if os.path.exists("../data/raw/hawker_centres.csv"):
    df_hawker = pd.read_csv("../data/raw/hawker_centres.csv")
    print(f"Already exists, loaded {len(df_hawker)} rows from disk")
else:
    hawker_url = "https://api-open.data.gov.sg/v1/public/api/datasets/d_4a086da0a5553be1d89383cd90d07ecd/poll-download"

    download_url = None
    for attempt in range(15):
        try:
            resp = requests.get(hawker_url, timeout=30)
            resp.raise_for_status()
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429:
                wait = 30 * (attempt + 1)
                print(f"Rate limited (429). Waiting {wait}s...")
                time.sleep(wait)
                continue
            raise
        data = resp.json()
        download_url = (data.get("data") or {}).get("url")
        if download_url:
            break
        print(f"Waiting for download URL... (attempt {attempt + 1})")
        time.sleep(3)

    if not download_url:
        raise RuntimeError("Could not get hawker centres download URL. Try again later.")

    hawker_resp = requests.get(download_url, timeout=30)
    hawker_resp.raise_for_status()

    try:
        hawker_geojson = hawker_resp.json()
        records = []
        for feature in hawker_geojson["features"]:
            props = feature["properties"]
            geom = feature["geometry"]
            coords = geom.get("coordinates", [None, None])
            if geom["type"] == "Point":
                lng, lat = coords[0], coords[1]
            elif geom["type"] == "Polygon":
                lngs = [c[0] for c in coords[0]]
                lats = [c[1] for c in coords[0]]
                lng, lat = sum(lngs) / len(lngs), sum(lats) / len(lats)
            else:
                lng, lat = None, None
            records.append({**props, "lat": lat, "lng": lng})
        df_hawker = pd.DataFrame(records)
    except (ValueError, KeyError):
        df_hawker = pd.read_csv(download_url)

    df_hawker.to_csv("../data/raw/hawker_centres.csv", index=False)

print(f"Shape: {df_hawker.shape}")
print(f"Columns: {list(df_hawker.columns)}")
df_hawker.head()

Already exists, loaded 129 rows from disk
Shape: (129, 22)
Columns: ['OBJECTID', 'LANDXADDRESSPOINT', 'LANDYADDRESSPOINT', 'ADDRESSBUILDINGNAME', 'ADDRESSPOSTALCODE', 'ADDRESSSTREETNAME', 'DESCRIPTION', 'NAME', 'PHOTOURL', 'ADDRESSBLOCKHOUSENUMBER', 'STATUS', 'AWARDED_DATE', 'IMPLEMENTATION_DATE', 'INFO_ON_CO_LOCATORS', 'ADDRESS_MYENV', 'EST_ORIGINAL_COMPLETION_DATE', 'HUP_COMPLETION_DATE', 'NUMBER_OF_COOKED_FOOD_STALLS', 'INC_CRC', 'FMEL_UPD_D', 'lat', 'lng']


,OBJECTID,LANDXADDRESSPOINT,LANDYADDRESSPOINT,ADDRESSBUILDINGNAME,ADDRESSPOSTALCODE,ADDRESSSTREETNAME,DESCRIPTION,NAME,PHOTOURL,ADDRESSBLOCKHOUSENUMBER,...,IMPLEMENTATION_DATE,INFO_ON_CO_LOCATORS,ADDRESS_MYENV,EST_ORIGINAL_COMPLETION_DATE,HUP_COMPLETION_DATE,NUMBER_OF_COOKED_FOOD_STALLS,INC_CRC,FMEL_UPD_D,lat,lng
0,119477,24332.824777,32135.918667,COMMONWEALTH CRESCENT MARKET,149644,Commonwealth Crescent,HUP Rebuilding,Commonwealth Crescent Market,http://www.nea.gov.sg/images/default-source/Ha...,31,...,NaN,NaN,"31, Commonwealth Crescent, Singapore 149644",1965,18/08/2004,39,40D5D8D44A44BFA9,20241211100036,1.306900,103.800367
1,119478,27873.457737,29690.583488,TIONG BAHRU MARKET,168898,Seng Poh Road,HUP Rebuilding,Tiong Bahru Market,http://www.nea.gov.sg/images/default-source/Ha...,30,...,NaN,NaN,"30, Seng Poh Road, Singapore 168898",NaN,10/02/2006,83,2AC5BFAA02772895,20241211100036,1.284786,103.832182
2,119479,31400.942558,31720.333016,GOLDEN MILE FOOD CENTRE,199583,Beach Road,HUP Standard Upgrading,Golden Mile Food Centre,http://www.nea.gov.sg/images/default-source/Ha...,505,...,NaN,NaN,"505, Beach Road, Singapore 199583",1975,04/07/2002,112,34F599BC64ED5790,20241211100036,1.303142,103.863878
3,119480,18273.882465,42119.284163,Heart of Yew Tee,680628,Yew Tee Close,New Centre,Yew Tee Hawker Centre,NaN,628,...,04/01/2023,hawker centre/flats/community club/polyclinic/...,NaN,2027,NaN,40,940576F260F3C469,20241211100036,1.397185,103.745922
4,119481,35624.169038,32414.399161,DUNMAN FOOD CENTRE,424768,Onan Road,HUP Standard Upgrading,Dunman Food Centre,http://www.nea.gov.sg/images/default-source/Ha...,271,...,NaN,NaN,"271, Onan Road, Singapore 424768",1974,03/12/2002,30,5B9777943F201078,20241211100036,1.309418,103.901825


## 6. HDB Car Park Information

Source: https://beta.data.gov.sg/collections/149/datasets/d_23f946fa557947f93a8043bbef41dd09/view

~2,300 HDB car parks with car park type, number of decks, and SVY21 coordinates. Note: coordinates are in SVY21 (Singapore's local projection), not WGS84. Conversion to lat/lng will be done in the feature engineering notebook.

In [23]:
if os.path.exists("../data/raw/hdb_carpark_info.csv"):
    df_carpark = pd.read_csv("../data/raw/hdb_carpark_info.csv")
    print(f"Already exists, loaded {len(df_carpark)} rows from disk")
else:
    df_carpark = fetch_datagov_csv(
        dataset_id="d_23f946fa557947f93a8043bbef41dd09",
        save_path="../data/raw/hdb_carpark_info.csv"
    )
print(f"Shape: {df_carpark.shape}")
print(f"Columns: {list(df_carpark.columns)}")
print(f"\nCar park types: {df_carpark['car_park_type'].value_counts().to_dict()}")
df_carpark.head()

Already exists, loaded 2266 rows from disk
Shape: (2266, 12)
Columns: ['car_park_no', 'address', 'x_coord', 'y_coord', 'car_park_type', 'type_of_parking_system', 'short_term_parking', 'free_parking', 'night_parking', 'car_park_decks', 'gantry_height', 'car_park_basement']

Car park types: {'MULTI-STOREY CAR PARK': 1125, 'SURFACE CAR PARK': 1071, 'BASEMENT CAR PARK': 46, 'SURFACE/MULTI-STOREY CAR PARK': 11, 'COVERED CAR PARK': 10, 'MECHANISED AND SURFACE CAR PARK': 2, 'MECHANISED CAR PARK': 1}


,car_park_no,address,x_coord,y_coord,car_park_type,type_of_parking_system,short_term_parking,free_parking,night_parking,car_park_decks,gantry_height,car_park_basement
0,ACB,BLK 270/271 ALBERT CENTRE BASEMENT CAR PARK,30314.7936,31490.4942,BASEMENT CAR PARK,ELECTRONIC PARKING,WHOLE DAY,NO,YES,1,1.8,Y
1,ACM,BLK 98A ALJUNIED CRESCENT,33758.4143,33695.5198,MULTI-STOREY CAR PARK,ELECTRONIC PARKING,WHOLE DAY,SUN & PH FR 7AM-10.30PM,YES,5,2.1,N
2,AH1,BLK 101 JALAN DUSUN,29257.7203,34500.3599,SURFACE CAR PARK,ELECTRONIC PARKING,WHOLE DAY,SUN & PH FR 7AM-10.30PM,YES,0,0.0,N
3,AK19,BLOCK 253 ANG MO KIO STREET 21,28185.4359,39012.6664,SURFACE CAR PARK,COUPON PARKING,7AM-7PM,NO,NO,0,0.0,N
4,AK31,BLK 302/348 ANG MO KIO STREET 31,29482.0290,38684.1754,SURFACE CAR PARK,COUPON PARKING,NO,NO,NO,0,0.0,N


## 7. Shopping Malls

Curated list of 86 major Singapore shopping malls with coordinates. Sourced from public data and manually verified.

In [24]:
df_malls = get_shopping_malls()
df_malls.to_csv("../data/raw/shopping_malls.csv", index=False)

print(f"Total malls: {len(df_malls)}")
df_malls.head()

Total malls: 87


,mall_name,lat,lng
0,VivoCity,1.264280,103.822117
1,ION Orchard,1.303920,103.831940
2,Plaza Singapura,1.300694,103.845278
3,Bugis Junction,1.299722,103.855278
4,Bugis+,1.300833,103.854722


## Summary

| Dataset | Rows | Source |
|---------|------|--------|
| HDB Resale Transactions | ~900k+ (2000-present, 4 datasets) | data.gov.sg |
| MRT/LRT Stations | ~190 | LTA DataMall shapefile (coords) + public records (opening dates) |
| Schools | ~346 | data.gov.sg |
| HDB Property Info | ~13k | data.gov.sg |
| Hawker Centres | ~100+ | data.gov.sg (GeoJSON) |
| HDB Car Park Info | ~2,300 | data.gov.sg |
| Shopping Malls | 86 | Curated list |

All raw datasets saved to `../data/raw/`. Next step: `1_eda.ipynb`.

In [25]:
import os
print("Files in ../data/raw/:")
for f in sorted(os.listdir("../data/raw")):
    size_mb = os.path.getsize(f"../data/raw/{f}") / (1024 * 1024)
    print(f"  {f:40s} {size_mb:.2f} MB")

Files in ../data/raw/:
  hawker_centres.csv                       0.05 MB
  hdb_carpark_info.csv                     0.29 MB
  hdb_property_info.csv                    0.90 MB
  mrt_lrt_stations.csv                     0.01 MB
  resale_transactions.csv                  59.65 MB
  resale_transactions_2000_2012.csv        29.42 MB
  resale_transactions_2012_2014.csv        4.17 MB
  resale_transactions_2015_2016.csv        3.07 MB
  resale_transactions_2017_onwards.csv     22.59 MB
  schools.csv                              0.13 MB
  shopping_malls.csv                       0.00 MB
